#### Faiss
Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [ ]:
from langchain_community.document_loaders import TextLoader # for loading text files
from langchain_community.vectorstores import FAISS #for creating vector store
from langchain_community.embeddings import OllamaEmbeddings #for creating embeddings 
from langchain_text_splitters import CharacterTextSplitter # for splitting text into chunks

loader=TextLoader("speech.txt") # loading the text file
documents=loader.load() # loading the documents
text_splitter=CharacterTextSplitter(chunk_size=1000,chunk_overlap=30) # creating a text splitter with chunk size of 1000 and overlap of 30
docs=text_splitter.split_documents(documents) # splitting the documents into chunks
 

In [ ]:
docs # printing the splitted documents

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
 Document(metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct our

In [ ]:
embeddings=OllamaEmbeddings(model="gemma:2b")  # creating an instance of OllamaEmbeddings to generate embeddings for the documents
db=FAISS.from_documents(docs,embeddings) # creating a FAISS vector store from the splitted documents and their corresponding embeddings
db # printing the FAISS vector store

In [ ]:
### querying 
query="How does the speaker describe the desired outcome of the war?" # defining a query to search for relevant information in the vector store
docs=db.similarity_search(query) # performing a similarity search in the vector store using the defined query to retrieve relevant documents
docs[0].page_content # printing the content of the most relevant document retrieved from the vector store based on the query


'…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

#### As a Retriever
We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers

In [ ]:
retriever=db.as_retriever() # creating a retriever from the FAISS vector store to facilitate retrieval of relevant documents based on queries
docs=retriever.invoke(query) # using the retriever to perform a search in the vector store based on the defined query to retrieve relevant documents
docs[0].page_content # printing the content of the most relevant document retrieved from the vector store based on the query using the retriever

#### Similarity Search with score
There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [ ]:
docs_and_score=db.similarity_search_with_score(query) # performing a similarity search in the vector store using the defined query to retrieve relevant documents along with their similarity scores
docs_and_score[0][0].page_content # printing the content of the most relevant document retrieved from the vector store based on the query along with its similarity score
docs_and_score[0][1] # printing the similarity score of the most relevant document retrieved from the vector store based on the query

In [ ]:
embedding_vector=embeddings.embed_query(query) # generating an embedding vector for the defined query using the OllamaEmbeddings instance
embedding_vector # printing the embedding vector generated for the defined query using the OllamaEmbeddings instance

In [ ]:
db.save_local("faiss_index") # saving the FAISS vector store locally to a directory named "faiss_index"
db2=FAISS.load_local("faiss_index",embeddings,allow_dangerous_deserialization=True) # loading the FAISS vector store from the local directory "faiss_index" using the same OllamaEmbeddings instance to ensure consistency in embeddings
db2 # printing the loaded FAISS vector store to verify that it has been loaded correctly